# Notebook 26: Prompt Tuning - The Most Efficient PEFT Method

**Learning Objectives:**
- Understand prompt tuning and soft prompt optimization
- Distinguish between prompt tuning and prefix tuning
- Learn about continuous (soft) vs discrete (hard) prompts
- Implement prompt tuning with <0.01% trainable parameters
- Understand when prompt tuning works and when it fails
- Compare prompt tuning vs LoRA vs full fine-tuning

**Prerequisites:** Notebooks 23-25 (LoRA, adapters)

## 1. Historical Context: From Discrete to Continuous Prompts

### Evolution of Prompting
```
2018-2020: In-Context Learning (ICL)
  Problem: GPT-3 shows impressive few-shot learning
  Method: Manually craft discrete text prompts
  Example: "Translate English to French: hello -> bonjour, goodbye -> "
  Issue: Sensitive to prompt wording, no optimization

2020: Prefix Tuning (Li & Liang)
  Problem: Discrete prompts suboptimal, hard to optimize
  Solution: Learn continuous "virtual tokens" prepended to input
  Key insight: Optimize embeddings directly, not discrete tokens
  Limitation: Separate prefixes for keys and values at each layer

2021: Prompt Tuning (Lester et al., Google)
  Problem: Prefix tuning requires many parameters
  Solution: Simplify - just learn soft prompt embeddings at input
  Key finding: Works as well as prefix tuning for large models (10B+)
  Result: <0.01% trainable parameters!

2022-2024: P-tuning v2, Prompt Tuning++
  Improvements: Better initialization, prompt ensembling
  But: Still limited to large models and similar tasks
```

### The Core Insight
```
Traditional Fine-Tuning:
  Update model weights to learn task
  Parameters: All model weights (millions/billions)

Prompt Engineering:
  Craft discrete text to guide model
  Parameters: 0 (but requires manual effort)

Prompt Tuning:
  Learn continuous prompt embeddings
  Parameters: Only prompt embeddings (hundreds)
  Best of both worlds!
```

### Discrete vs Continuous Prompts
```
Discrete Prompt (traditional):
  Text: "Classify sentiment: [INPUT] Sentiment:"
  Tokens: [CLS] Classify sentiment : [INPUT] Sentiment :
  Embeddings: E("Classify"), E("sentiment"), ...
  Problem: Limited to vocabulary, discrete optimization hard

Continuous Prompt (prompt tuning):
  Learnable: [P1] [P2] [P3] [INPUT]
  Where P1, P2, P3 are continuous vectors in embedding space
  Not tied to vocabulary tokens!
  Gradient descent optimizes them directly
```

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, List, Dict, Tuple
import math

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 2. Prompt Tuning vs Prefix Tuning: Key Differences

### Prefix Tuning (2020)
```
Architecture:
  Layer 1: K1 = concat([P_k1], K1_frozen)
           V1 = concat([P_v1], V1_frozen)
  Layer 2: K2 = concat([P_k2], K2_frozen)
           V2 = concat([P_v2], V2_frozen)
  ...
  Layer L: KL = concat([P_kL], KL_frozen)
           VL = concat([P_vL], VL_frozen)

Parameters: 2 × L × n_prefix × d_model
  L = number of layers
  n_prefix = prefix length (e.g., 20)
  d_model = hidden dim (e.g., 768)
  Example: 2 × 12 × 20 × 768 = 368,640 params (~0.3%)
```

### Prompt Tuning (2021)
```
Architecture:
  Input: X = concat([P1, P2, ..., Pn], X_tokens)
  Where Pi are learnable embeddings
  Rest of model: completely frozen

Parameters: n_prompt × d_model
  n_prompt = prompt length (e.g., 20)
  d_model = hidden dim (e.g., 768)
  Example: 20 × 768 = 15,360 params (~0.01%)
```

### Comparison
| Aspect | Prefix Tuning | Prompt Tuning |
|--------|---------------|---------------|
| Parameters | 2 × L × n × d | n × d |
| Typical % | 0.1-1% | <0.01% |
| Modification | Keys & values per layer | Input only |
| Complexity | Higher | Lower |
| Performance (10B+) | Excellent | Excellent |
| Performance (<1B) | Good | Poor |
| Implementation | More complex | Very simple |

Key finding from Lester et al. (2021):
```
Model Size vs Prompt Tuning Effectiveness:
  60M params:  Prompt tuning << Full FT (gap: 30%)
  250M params: Prompt tuning < Full FT (gap: 15%)
  1B params:   Prompt tuning ≈ Full FT (gap: 5%)
  11B params:  Prompt tuning ≈ Full FT (gap: <2%)

Conclusion: Prompt tuning is MODEL-SIZE dependent!
Only use for models >10B parameters.
```

In [ ]:
class PromptTuning(nn.Module):
    """Soft prompt tuning - learnable continuous prompts"""
    
    def __init__(
        self,
        n_prompt_tokens: int = 20,
        d_model: int = 768,
        init_method: str = 'random',  # 'random', 'vocab', 'class_label'
        vocab_embeddings: Optional[torch.Tensor] = None
    ):
        """
        Args:
            n_prompt_tokens: Number of soft prompt tokens
            d_model: Model embedding dimension
            init_method: How to initialize soft prompts
            vocab_embeddings: Pretrained embeddings for initialization
        """
        super().__init__()
        self.n_prompt_tokens = n_prompt_tokens
        self.d_model = d_model
        
        # Learnable soft prompt embeddings
        self.soft_prompt = nn.Parameter(
            torch.randn(n_prompt_tokens, d_model)
        )
        
        # Initialize
        self._initialize(init_method, vocab_embeddings)
    
    def _initialize(self, method: str, vocab_embeddings: Optional[torch.Tensor]):
        """Initialize soft prompts"""
        if method == 'random':
            # Random initialization (uniform)
            nn.init.uniform_(self.soft_prompt, -0.5, 0.5)
        
        elif method == 'vocab' and vocab_embeddings is not None:
            # Initialize from random vocabulary embeddings
            # Sample random tokens from vocab
            vocab_size = vocab_embeddings.shape[0]
            random_indices = torch.randint(0, vocab_size, (self.n_prompt_tokens,))
            self.soft_prompt.data = vocab_embeddings[random_indices].clone()
        
        elif method == 'class_label' and vocab_embeddings is not None:
            # Initialize from class label tokens (e.g., "positive", "negative")
            # This is a simplification - in practice, use actual label token IDs
            self.soft_prompt.data = vocab_embeddings[:self.n_prompt_tokens].clone()
    
    def forward(self, input_embeddings: torch.Tensor) -> torch.Tensor:
        """
        Prepend soft prompts to input embeddings
        
        Args:
            input_embeddings: (batch, seq_len, d_model)
        Returns:
            prompted_embeddings: (batch, n_prompt + seq_len, d_model)
        """
        batch_size = input_embeddings.shape[0]
        
        # Expand soft prompt for batch
        # (n_prompt, d_model) -> (batch, n_prompt, d_model)
        soft_prompt_batch = self.soft_prompt.unsqueeze(0).expand(
            batch_size, -1, -1
        )
        
        # Concatenate prompt with input
        prompted_embeddings = torch.cat(
            [soft_prompt_batch, input_embeddings], dim=1
        )
        
        return prompted_embeddings
    
    def num_parameters(self) -> int:
        return self.n_prompt_tokens * self.d_model


class PrefixTuning(nn.Module):
    """Prefix tuning - learnable prefixes for keys and values"""
    
    def __init__(
        self,
        n_layers: int = 12,
        n_prefix_tokens: int = 20,
        d_model: int = 768,
        n_heads: int = 12
    ):
        super().__init__()
        self.n_layers = n_layers
        self.n_prefix_tokens = n_prefix_tokens
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        # Learnable prefix for each layer (keys and values)
        # Shape: (n_layers, 2, n_heads, n_prefix, d_head)
        # 2 for keys and values
        self.prefix = nn.Parameter(
            torch.randn(n_layers, 2, n_heads, n_prefix_tokens, self.d_head)
        )
        
        # Initialize
        nn.init.normal_(self.prefix, std=0.02)
    
    def get_prefix(self, layer_idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Get prefix keys and values for a specific layer
        
        Args:
            layer_idx: Layer index (0 to n_layers-1)
        Returns:
            prefix_keys: (n_heads, n_prefix, d_head)
            prefix_values: (n_heads, n_prefix, d_head)
        """
        prefix_keys = self.prefix[layer_idx, 0]  # (n_heads, n_prefix, d_head)
        prefix_values = self.prefix[layer_idx, 1]
        return prefix_keys, prefix_values
    
    def num_parameters(self) -> int:
        return (self.n_layers * 2 * self.n_heads * 
                self.n_prefix_tokens * self.d_head)


# Compare parameter counts
d_model = 768
n_layers = 12
n_heads = 12
n_tokens = 20

prompt_tuning = PromptTuning(n_tokens, d_model)
prefix_tuning = PrefixTuning(n_layers, n_tokens, d_model, n_heads)

# Base model params (BERT-base approximate)
base_params = 110_000_000

print("Parameter Comparison: Prompt Tuning vs Prefix Tuning\n")
print(f"Base model: {base_params:,} parameters\n")

prompt_params = prompt_tuning.num_parameters()
prefix_params = prefix_tuning.num_parameters()

print(f"Prompt Tuning:")
print(f"  Parameters: {prompt_params:,}")
print(f"  % of base:  {100 * prompt_params / base_params:.4f}%")
print(f"  Formula:    n_tokens × d_model = {n_tokens} × {d_model}\n")

print(f"Prefix Tuning:")
print(f"  Parameters: {prefix_params:,}")
print(f"  % of base:  {100 * prefix_params / base_params:.4f}%")
print(f"  Formula:    n_layers × 2 × n_heads × n_prefix × d_head")
print(f"              = {n_layers} × 2 × {n_heads} × {n_tokens} × {d_model//n_heads}\n")

print(f"Prefix tuning uses {prefix_params / prompt_params:.1f}x more parameters!")

# Test forward pass
batch_size = 4
seq_len = 128
input_emb = torch.randn(batch_size, seq_len, d_model)

prompted_emb = prompt_tuning(input_emb)
print(f"\nPrompt Tuning Forward Pass:")
print(f"  Input shape:  {input_emb.shape}")
print(f"  Output shape: {prompted_emb.shape}")
print(f"  Added {n_tokens} soft prompt tokens to input")

## 3. Continuous Prompts: Not Tied to Vocabulary!

### The Key Insight
```
Discrete prompt:
  "Classify the sentiment of this review:"
  ↓ (tokenize)
  ["Classify", "the", "sentiment", "of", "this", "review", ":"]
  ↓ (embed)
  [E_vocab[101], E_vocab[234], E_vocab[445], ...]
  
  Constraint: Must be valid vocabulary tokens!
  Problem: Vocabulary may not contain optimal "meta-tokens" for task

Continuous (soft) prompt:
  [P1, P2, P3, P4, P5, P6, P7]
  Where each Pi ∈ ℝ^d is a learnable vector
  
  Freedom: Not constrained to vocabulary!
  Pi can be anywhere in embedding space
  Might not correspond to any real word
  Learned via gradient descent
```

### Visualization Analogy
```
Embedding Space (simplified 2D):

  ^                 
  |    cat •              • dog
  |         
  |              • happy
  |  
  |                    • P2  <-- Soft prompt (not a real word!)
  |        • sad
  |                        
  |  • angry    • P1  <-- Soft prompt
  +-------------------------------->

Discrete prompts: Must use actual tokens (cat, dog, happy, sad, ...)
Soft prompts: Can be ANYWHERE in space (P1, P2 between words)
```

### What Do Soft Prompts Learn?
Research findings (Lester et al., 2021):
1. **Not interpretable**: Soft prompts don't correspond to readable text
2. **Task-specific**: Different tasks learn different prompt geometries
3. **Position matters**: First few tokens most important
4. **Initialization helps**: Starting from vocabulary better than random
5. **Length matters**: 20-100 tokens typical (more doesn't always help)

In [ ]:
# Demonstrate that soft prompts aren't vocabulary tokens
def analyze_soft_prompts(model: PromptTuning, vocab_embeddings: torch.Tensor):
    """
    Analyze how far soft prompts are from vocabulary embeddings
    """
    soft_prompts = model.soft_prompt.data  # (n_prompt, d_model)
    n_prompt = soft_prompts.shape[0]
    
    # Find nearest vocabulary token for each soft prompt
    nearest_distances = []
    
    for i in range(n_prompt):
        soft_prompt_i = soft_prompts[i]  # (d_model,)
        
        # Compute distances to all vocabulary embeddings
        distances = torch.norm(
            vocab_embeddings - soft_prompt_i.unsqueeze(0), 
            dim=1
        )
        
        # Find nearest
        min_dist = distances.min().item()
        nearest_distances.append(min_dist)
    
    return nearest_distances


# Create synthetic vocabulary embeddings
vocab_size = 30000
d_model = 512
vocab_embeddings = torch.randn(vocab_size, d_model)
vocab_embeddings = F.normalize(vocab_embeddings, dim=1)  # Normalize

# Create two models: one initialized randomly, one from vocab
model_random = PromptTuning(20, d_model, init_method='random')
model_vocab = PromptTuning(20, d_model, init_method='vocab', 
                           vocab_embeddings=vocab_embeddings)

# Normalize for fair comparison
model_random.soft_prompt.data = F.normalize(model_random.soft_prompt.data, dim=1)

# Analyze
dist_random = analyze_soft_prompts(model_random, vocab_embeddings)
dist_vocab = analyze_soft_prompts(model_vocab, vocab_embeddings)

print("Distance from Soft Prompts to Nearest Vocabulary Token:\n")
print(f"Random initialization:")
print(f"  Mean distance: {np.mean(dist_random):.4f}")
print(f"  Min distance:  {np.min(dist_random):.4f}")
print(f"  Max distance:  {np.max(dist_random):.4f}\n")

print(f"Vocabulary initialization:")
print(f"  Mean distance: {np.mean(dist_vocab):.4f}")
print(f"  Min distance:  {np.min(dist_vocab):.4f}")
print(f"  Max distance:  {np.max(dist_vocab):.4f}\n")

# Simulate training (soft prompts drift from vocabulary)
# After training, soft prompts optimize for task, not vocab proximity
def simulate_training(model, n_steps=100):
    distances_over_time = []
    for step in range(n_steps):
        # Simulate gradient update (random walk away from init)
        with torch.no_grad():
            model.soft_prompt += torch.randn_like(model.soft_prompt) * 0.01
            model.soft_prompt.data = F.normalize(model.soft_prompt.data, dim=1)
        
        # Measure distance to vocabulary
        dists = analyze_soft_prompts(model, vocab_embeddings)
        distances_over_time.append(np.mean(dists))
    
    return distances_over_time

# Simulate
model_test = PromptTuning(20, d_model, init_method='vocab', 
                          vocab_embeddings=vocab_embeddings)
distances_trajectory = simulate_training(model_test)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Distance comparison
positions = np.arange(len(dist_random))
width = 0.35

ax1.bar(positions - width/2, dist_random, width, label='Random Init', 
        color='#FF6B6B', alpha=0.7)
ax1.bar(positions + width/2, dist_vocab, width, label='Vocab Init',
        color='#4ECDC4', alpha=0.7)
ax1.set_xlabel('Soft Prompt Token Index', fontsize=11)
ax1.set_ylabel('Distance to Nearest Vocab Token', fontsize=11)
ax1.set_title('Soft Prompts vs Vocabulary: Distance Analysis', 
              fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Training trajectory
ax2.plot(distances_trajectory, linewidth=2, color='#4ECDC4')
ax2.axhline(y=distances_trajectory[0], color='#FF6B6B', 
            linestyle='--', label='Initial distance')
ax2.set_xlabel('Training Steps (simulated)', fontsize=11)
ax2.set_ylabel('Mean Distance to Vocabulary', fontsize=11)
ax2.set_title('Soft Prompts Drift from Vocabulary During Training',
              fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight: Soft prompts start near vocabulary tokens but")
print("drift away during training, optimizing for task performance")
print("rather than staying aligned with discrete tokens.")

## 4. Implementation with PEFT Library

### Prompt Tuning in Practice

In [ ]:
# Example code structure (requires PEFT library)
"""
from transformers import AutoModelForSequenceClassification
from peft import (
    get_peft_model, 
    PromptTuningConfig, 
    TaskType,
    PromptTuningInit
)

# Load base model
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-large",  # 355M params - prompt tuning needs large models!
    num_labels=2
)

# Configure prompt tuning
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20,  # Number of soft prompt tokens
    prompt_tuning_init=PromptTuningInit.RANDOM,  # or TEXT, VOCAB_SAMPLE
    prompt_tuning_init_text="Classify if this is positive or negative:",  # if TEXT
    tokenizer_name_or_path="roberta-large",
)

# Apply prompt tuning
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
# Output: trainable params: 15,360 || all params: 355,015,360 || trainable%: 0.0043

# Train as normal
# Only soft prompt embeddings are updated!
# ...

# Save (only saves prompt embeddings)
model.save_pretrained("./sentiment_prompt")
# Only ~60KB file!
"""

print("Prompt Tuning with PEFT (see code comments)\n")
print("Key configuration options:")
print("  num_virtual_tokens: Soft prompt length (20-100 typical)")
print("  prompt_tuning_init: RANDOM, TEXT, or VOCAB_SAMPLE")
print("    - RANDOM: Random initialization")
print("    - TEXT: Initialize from actual text prompt")
print("    - VOCAB_SAMPLE: Sample random vocabulary embeddings")
print("\nBest practice: TEXT init with task description often works best")

## 5. When Prompt Tuning Works (and When It Fails)

### Success Criteria

#### 1. Model Size (CRITICAL)
```
Empirical results from Lester et al. (2021):

Model Size    | Prompt Tuning Performance
--------------|---------------------------
< 100M params | Poor (30-50% of full FT)
100M - 1B     | Okay (70-85% of full FT)
1B - 10B      | Good (90-95% of full FT)
> 10B params  | Excellent (95-100% of full FT)

Recommendation: Only use prompt tuning for models ≥ 1B parameters
```

#### 2. Task Similarity to Pretraining
```
Works well:
  - Sentiment analysis (common in pretraining)
  - Question answering (seen during pretraining)
  - Text classification (general capability)

Works poorly:
  - Highly specialized domains (medical, legal)
  - Tasks requiring new knowledge
  - Tasks with very different data distribution
```

#### 3. Dataset Size
```
Small datasets (< 1000 examples):
  - Prompt tuning often better than full FT
  - Less prone to overfitting
  - Only 15K parameters to overfit vs millions

Large datasets (> 100K examples):
  - Full FT or LoRA often better
  - More data enables learning complex patterns
  - Prompt capacity becomes bottleneck
```

#### 4. Task Complexity
```
Simple tasks:
  - Binary classification ✓
  - Multi-class classification ✓
  - Simple extraction ✓

Complex tasks:
  - Sequence-to-sequence ✗
  - Complex reasoning ✗
  - Multi-step problems ✗
```

### When to Use Each Method

**Use Prompt Tuning when:**
- Model is very large (≥10B)
- Dataset is small (<1K examples)
- Task is simple classification
- Extreme parameter efficiency needed
- Multiple tasks on same base model

**Use LoRA when:**
- Model is medium-large (1B-10B)
- Dataset is medium (1K-100K)
- Task is moderately complex
- Need good efficiency + performance
- Production deployment

**Use Full FT when:**
- Model is small (<1B)
- Dataset is large (>100K)
- Task is complex
- Maximum performance needed
- Resources not constrained

In [ ]:
# Simulate performance vs model size
model_sizes = np.array([60, 250, 750, 3000, 11000, 50000])  # Million parameters
model_names = ['60M', '250M', '750M', '3B', '11B', '50B']

# Performance relative to full fine-tuning (synthetic but realistic)
prompt_tuning_perf = np.array([0.45, 0.65, 0.85, 0.92, 0.98, 0.99])
lora_perf = np.array([0.85, 0.90, 0.93, 0.95, 0.97, 0.98])
full_ft_perf = np.ones(len(model_sizes))  # 100% by definition

# Efficiency (inverse of trainable parameters, normalized)
prompt_tuning_eff = np.array([100, 100, 100, 100, 100, 100])  # <0.01%
lora_eff = np.array([20, 20, 20, 20, 20, 20])  # ~0.1-1%
full_ft_eff = np.array([1, 1, 1, 1, 1, 1])  # 100% (baseline)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Performance vs model size
ax1.plot(model_sizes, prompt_tuning_perf * 100, 'o-', 
         linewidth=2.5, markersize=8, label='Prompt Tuning', color='#FF6B6B')
ax1.plot(model_sizes, lora_perf * 100, 's-',
         linewidth=2.5, markersize=8, label='LoRA', color='#4ECDC4')
ax1.plot(model_sizes, full_ft_perf * 100, '^-',
         linewidth=2.5, markersize=8, label='Full Fine-Tuning', color='#95E1D3')

ax1.set_xscale('log')
ax1.set_xlabel('Model Size (Million Parameters, log scale)', fontsize=12)
ax1.set_ylabel('Performance (% of Full FT)', fontsize=12)
ax1.set_title('PEFT Performance vs Model Size', fontsize=14, fontweight='bold')
ax1.set_xticks(model_sizes)
ax1.set_xticklabels(model_names)
ax1.set_ylim([40, 105])
ax1.axhline(y=90, color='gray', linestyle='--', alpha=0.5, label='90% threshold')
ax1.axvline(x=1000, color='orange', linestyle='--', alpha=0.5, 
            label='1B param threshold')
ax1.legend(fontsize=10, loc='lower right')
ax1.grid(True, alpha=0.3)

# Add annotation
ax1.annotate('Prompt tuning only effective\nat large scale',
             xy=(3000, 92), xytext=(500, 75),
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
             fontsize=10, color='red', fontweight='bold')

# Efficiency-performance tradeoff
# Plot for 11B model (where prompt tuning is competitive)
methods = ['Prompt\nTuning', 'LoRA', 'Full FT']
performance_11b = [prompt_tuning_perf[4] * 100, 
                   lora_perf[4] * 100, 
                   full_ft_perf[4] * 100]
efficiency = [prompt_tuning_eff[4], lora_eff[4], full_ft_eff[4]]
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3']

# Scatter plot
for i, (perf, eff, method, color) in enumerate(zip(performance_11b, efficiency, 
                                                     methods, colors)):
    ax2.scatter(eff, perf, s=500, alpha=0.6, color=color, edgecolors='black', 
                linewidths=2, label=method, zorder=3)
    ax2.annotate(method, xy=(eff, perf), xytext=(0, -30),
                textcoords='offset points', ha='center', fontsize=10,
                fontweight='bold')

ax2.set_xscale('log')
ax2.set_xlabel('Parameter Efficiency (higher = fewer params)', fontsize=12)
ax2.set_ylabel('Performance (% of Full FT)', fontsize=12)
ax2.set_title('Efficiency-Performance Tradeoff (11B Model)', 
              fontsize=14, fontweight='bold')
ax2.set_xlim([0.5, 200])
ax2.set_ylim([95, 100.5])
ax2.grid(True, alpha=0.3)

# Add Pareto frontier
ax2.plot([1, 20, 100], [100, 97, 98], 'k--', alpha=0.3, linewidth=1.5, 
         label='Pareto frontier')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("1. Prompt tuning only competitive for models >1B parameters")
print("2. For large models (>10B), prompt tuning approaches full FT performance")
print("3. LoRA provides better performance for small-medium models")
print("4. Prompt tuning offers best parameter efficiency when it works")

## 6. Comparison: Prompt Tuning vs LoRA vs Full FT

### Comprehensive Comparison Table

| Metric | Prompt Tuning | LoRA | Full FT |
|--------|---------------|------|----------|
| **Trainable Parameters** | <0.01% | 0.1-1% | 100% |
| **Typical Param Count** (BERT-base) | ~15K | ~300K | ~110M |
| **Memory (Training)** | Very Low | Low | High |
| **Training Speed** | Fast | Fast | Slow |
| **Inference Speed** | Same as base | Same as base | Same as base |
| **Performance (Small Model <1B)** | Poor | Good | Best |
| **Performance (Large Model >10B)** | Excellent | Excellent | Best |
| **Small Dataset (<1K)** | Good (less overfit) | Good | Poor (overfit) |
| **Large Dataset (>100K)** | Okay | Good | Best |
| **Simple Tasks** | Excellent | Excellent | Excellent |
| **Complex Tasks** | Poor | Good | Best |
| **Implementation Complexity** | Very Simple | Simple | Simple |
| **Multi-Task Efficiency** | Excellent | Good | Poor |
| **Interpretability** | None | Low | Low |
| **When to Use** | Very large models, simple tasks | Most scenarios | Max performance, resources available |

### Storage Requirements Comparison
```
For 10 tasks on BERT-base (110M params):

Full FT: 10 × 110M = 1.1B params = 4.4 GB
LoRA:    1 × 110M + 10 × 300K = 113M params = 452 MB
Prompt:  1 × 110M + 10 × 15K = 110.15M params = 440.6 MB

Prompt Tuning saves: 3.96 GB (90% reduction)
```

### Decision Matrix
```
Model Size    | Small Dataset | Large Dataset | Complex Task
--------------|---------------|---------------|--------------
< 1B params   | LoRA          | Full FT       | Full FT
1B - 10B      | Prompt/LoRA   | LoRA          | LoRA
> 10B params  | Prompt Tuning | Prompt/LoRA   | LoRA/Full FT
```

In [ ]:
# Comprehensive comparison visualization
import pandas as pd

# Create comparison data
comparison_data = {
    'Metric': [
        'Parameters\n(BERT-base)',
        'Storage\n(10 tasks)',
        'Training\nMemory',
        'Small Model\nPerformance',
        'Large Model\nPerformance',
        'Small Dataset\nFit',
        'Implementation\nComplexity',
        'Multi-Task\nEfficiency'
    ],
    'Prompt Tuning': [10, 10, 9, 3, 9, 9, 10, 10],
    'LoRA': [8, 8, 8, 8, 9, 8, 8, 8],
    'Full FT': [1, 1, 1, 10, 10, 4, 7, 1]
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Radar chart
ax1 = plt.subplot(221, projection='polar')
categories = comparison_data['Metric']
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

for method, color in [('Prompt Tuning', '#FF6B6B'), 
                      ('LoRA', '#4ECDC4'), 
                      ('Full FT', '#95E1D3')]:
    values = comparison_data[method] + comparison_data[method][:1]
    ax1.plot(angles, values, 'o-', linewidth=2, label=method, color=color)
    ax1.fill(angles, values, alpha=0.15, color=color)

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories, size=8)
ax1.set_ylim(0, 10)
ax1.set_yticks([2, 4, 6, 8, 10])
ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax1.set_title('Multi-Dimensional Comparison', fontsize=12, fontweight='bold', pad=20)
ax1.grid(True)

# 2. Parameter count comparison
ax2 = plt.subplot(222)
methods = ['Prompt\nTuning', 'LoRA', 'Full FT']
param_counts = [15_360, 294_912, 110_000_000]  # BERT-base
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3']

bars = ax2.bar(methods, param_counts, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_yscale('log')
ax2.set_ylabel('Trainable Parameters (log scale)', fontsize=11)
ax2.set_title('Parameter Efficiency Comparison', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar, val in zip(bars, param_counts):
    height = bar.get_height()
    if val < 1_000_000:
        label = f'{val/1000:.1f}K'
    else:
        label = f'{val/1_000_000:.1f}M'
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Performance vs model size
ax3 = plt.subplot(223)
model_sizes_mb = [60, 250, 750, 3000, 11000]
model_labels = ['60M', '250M', '750M', '3B', '11B']

prompt_perf = [45, 65, 85, 92, 98]
lora_perf = [85, 90, 93, 95, 97]
ft_perf = [100, 100, 100, 100, 100]

ax3.plot(range(len(model_labels)), prompt_perf, 'o-', 
         linewidth=2.5, markersize=8, label='Prompt Tuning', color='#FF6B6B')
ax3.plot(range(len(model_labels)), lora_perf, 's-',
         linewidth=2.5, markersize=8, label='LoRA', color='#4ECDC4')
ax3.plot(range(len(model_labels)), ft_perf, '^-',
         linewidth=2.5, markersize=8, label='Full FT', color='#95E1D3')

ax3.set_xticks(range(len(model_labels)))
ax3.set_xticklabels(model_labels)
ax3.set_xlabel('Model Size', fontsize=11)
ax3.set_ylabel('Performance (% of Full FT)', fontsize=11)
ax3.set_title('Performance vs Model Size', fontsize=12, fontweight='bold')
ax3.axhline(y=90, color='gray', linestyle='--', alpha=0.5)
ax3.legend()
ax3.grid(alpha=0.3)
ax3.set_ylim([40, 105])

# 4. Decision tree (text-based)
ax4 = plt.subplot(224)
ax4.axis('off')

decision_tree = """
PEFT Method Selection Guide
================================

Model Size < 1B?
  YES → Use LoRA or Full FT
        Prompt tuning performs poorly
  NO  → Continue...

Model Size > 10B?
  YES → Consider Prompt Tuning
        Excellent efficiency, good performance
  NO  → LoRA recommended (1B-10B range)

Task Complexity?
  Simple (classification):
    → Prompt Tuning (if large model)
  Complex (seq2seq, reasoning):
    → LoRA or Full FT

Dataset Size?
  Small (<1K examples):
    → Prompt Tuning or LoRA
       (avoid overfitting)
  Large (>100K examples):
    → LoRA or Full FT
       (can leverage more capacity)

Multiple Tasks?
  YES → Prompt Tuning
        Minimal storage per task
  NO  → Any method works
"""

ax4.text(0.1, 0.5, decision_tree, fontsize=9, family='monospace',
         verticalalignment='center',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
ax4.set_title('Method Selection Decision Tree', fontsize=12, 
              fontweight='bold', loc='center', pad=20)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("RECOMMENDATION SUMMARY")
print("="*70)
print("\nPrompt Tuning: Best for very large models (>10B), simple tasks")
print("  Pros: Extreme efficiency (<0.01%), minimal storage")
print("  Cons: Only works at scale, limited to simple tasks\n")
print("LoRA: Best for most scenarios (1B-10B models)")
print("  Pros: Good balance, works across model sizes")
print("  Cons: More parameters than prompt tuning\n")
print("Full FT: Best for maximum performance")
print("  Pros: Best possible performance")
print("  Cons: High resource requirements, storage")
print("="*70)

## Summary: Prompt Tuning

### Key Takeaways

1. **Core Concept**
   - Learn continuous "soft prompts" instead of discrete text
   - Soft prompts = learnable embeddings prepended to input
   - Only optimize these embeddings, freeze entire model
   - Most parameter-efficient PEFT method: <0.01%

2. **vs Prefix Tuning**
   - Prompt tuning: Only input layer (n × d params)
   - Prefix tuning: All layers (2 × L × n × d params)
   - Prompt tuning 10-20x more efficient
   - Similar performance for large models (>10B)

3. **Continuous vs Discrete Prompts**
   - Discrete: Limited to vocabulary tokens
   - Continuous: Anywhere in embedding space
   - Continuous optimized via gradient descent
   - Not human-interpretable, but task-optimal

4. **When It Works**
   - Large models (>10B parameters) ✓
   - Simple tasks (classification) ✓
   - Small datasets (<1K examples) ✓
   - Multiple tasks on same base ✓

5. **When It Fails**
   - Small models (<1B) - 30-50% performance drop
   - Complex tasks (seq2seq, reasoning)
   - Very different task distribution from pretraining
   - Need maximum performance

6. **Comparison with Other Methods**
   - Most efficient: Prompt tuning (15K params)
   - Best versatility: LoRA (300K params)
   - Best performance: Full FT (110M params)
   - Best for large models: Prompt tuning ≈ LoRA
   - Best for small models: LoRA > Prompt tuning

7. **Practical Recommendations**
   ```
   Use Prompt Tuning when:
   - Model ≥ 10B params
   - Simple classification task
   - Extreme efficiency required
   - Many tasks on same base
   
   Otherwise use LoRA:
   - More reliable across scales
   - Better for complex tasks
   - Still very efficient (0.1-1%)
   ```

### Implementation Checklist
- [ ] Verify model size (>10B recommended)
- [ ] Choose prompt length (20-100 tokens typical)
- [ ] Initialize from text description (best practice)
- [ ] Use small learning rate (1e-3 to 1e-5)
- [ ] Monitor for underfitting (sign of insufficient capacity)
- [ ] Compare with LoRA baseline

### Next Steps
- Notebook 27: Head-to-head PEFT benchmark
- Notebook 28: LoRA merging strategies
- Notebook 29: Advanced LoRA variants

### Further Reading
- Lester et al. (2021): "The Power of Scale for Parameter-Efficient Prompt Tuning"
- Li & Liang (2021): "Prefix-Tuning: Optimizing Continuous Prompts"
- Liu et al. (2022): "P-Tuning v2: Prompt Tuning Can Be Comparable to Fine-tuning"
- PEFT library: https://github.com/huggingface/peft